# Test Suite — IST 488/688 Final Project
**Master test runner for all four members' modules.**

This notebook is **self-contained** — it inlines (or stubs) the key functions from each teammate's notebook and tests them against `test_fixtures.json`. Anyone on the team can run this to verify the pipeline works without needing the other notebooks loaded.

## Setup
1. Upload `test_fixtures.json` to Colab (drag into the file panel, or place in working dir).
2. Set `OPENAI_API_KEY` in Colab Secrets if you want to run the OpenAI-dependent tests:
   ```python
   from google.colab import userdata
   import os
   os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
   ```
3. Run cells top to bottom. Tests print `✓` (pass) or `✗` (fail) inline. A summary prints at the end.

## What's tested

| Section | Owner | Function(s) tested | Needs API key? |
|---|---|---|---|
| §3 | Lauren | `normalize_place`, schema validation | No |
| §4 | Ryan | `_extract_html_text`, `_find_menu_links`, `fetch_restaurant_content` (cache path) | No |
| §5 | Leytisha | `structure_record`, `simple_retrieve`, ethics rubric, self-reflection | Mixed |
| §6 | Toby | `build_embedding_text`, `chroma_search` end-to-end | No (uses local embeddings) |
| §7 | All | Full pipeline integration | Yes (OpenAI) |


## 1. Install dependencies

In [ ]:
!pip install -q requests beautifulsoup4 trafilatura pdfplumber lxml chromadb sentence-transformers openai


## 2. Load fixtures + setup

In [ ]:
import os
import json
import hashlib
from io import BytesIO
from pathlib import Path

# --- Load fixtures ---
F = json.loads(Path("test_fixtures.json").read_text())
print(f"Loaded fixtures. Top-level keys:")
for k in F:
    if not k.startswith("_"):
        print(f"  • {k}")

# --- Test result tracker ---
RESULTS = []
def record(name, passed, detail=""):
    RESULTS.append((name, passed, detail))
    icon = "✓" if passed else "✗"
    line = f"  {icon} {name}"
    if detail:
        line += f"  — {detail}"
    print(line)


## 3. Lauren — Discovery Agent tests

Tests `normalize_place()` against realistic Places API responses. No API key needed.


In [ ]:
# --- Inline copy of Lauren's normalize_place (so this notebook is self-contained) ---
def normalize_place(raw: dict, city: str) -> dict:
    return {
        "place_id": raw.get("id", ""),
        "name": (raw.get("displayName") or {}).get("text", ""),
        "address": raw.get("formattedAddress", ""),
        "city": city.split(",")[0].strip(),
        "rating": raw.get("rating"),
        "user_rating_count": raw.get("userRatingCount"),
        "price_level": raw.get("priceLevel"),
        "website_url": raw.get("websiteUri"),
        "types": raw.get("types", []),
    }

print("§3 — LAUREN")
raw_resp = F["raw_places_api_responses"]["syracuse_response"]["places"]

# Test 3.1: normalize_place returns canonical shape
sample = normalize_place(raw_resp[0], "Syracuse, NY")
required = ["place_id", "name", "address", "city", "rating", "website_url", "types"]
missing = [k for k in required if k not in sample]
record("3.1 normalize_place returns canonical schema", not missing,
       f"missing: {missing}" if missing else "all fields present")

# Test 3.2: city is parsed correctly
record("3.2 city parsed from 'Syracuse, NY' → 'Syracuse'", sample["city"] == "Syracuse",
       f"got {sample['city']!r}")

# Test 3.3: displayName.text extracted, not the whole dict
record("3.3 displayName extracted as string", isinstance(sample["name"], str) and sample["name"] == "Dinosaur Bar-B-Que",
       f"got {sample['name']!r}")

# Test 3.4: places without websiteUri get None for website_url (so Ryan can filter them)
no_site = normalize_place(raw_resp[2], "Syracuse, NY")
record("3.4 missing websiteUri → website_url is None/falsy", not no_site["website_url"],
       f"got {no_site['website_url']!r}")


## 4. Ryan — Scraper tests

Tests HTML extraction, menu-link discovery, and the cache layer. No network calls (uses fixture HTML).


In [ ]:
import requests
from bs4 import BeautifulSoup
import trafilatura
from urllib.parse import urljoin, urlparse
import re
import logging
logging.basicConfig(level=logging.WARNING)

MENU_LINK_PATTERNS = [r"menu", r"menus", r"food", r"dinner", r"lunch", r"brunch", r"drinks"]
CACHE_DIR = Path("./scrape_cache_test"); CACHE_DIR.mkdir(exist_ok=True)
def _cache_path(url): return CACHE_DIR / f"{hashlib.md5(url.encode()).hexdigest()}.json"

def _extract_html_text(html: str) -> str:
    try:
        extracted = trafilatura.extract(html, include_tables=True, include_links=False, no_fallback=False)
        if extracted and len(extracted) > 200:
            return extracted
    except Exception:
        pass
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()
    return re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n", strip=True))

def _find_menu_links(html: str, base_url: str) -> list:
    soup = BeautifulSoup(html, "lxml")
    base_domain = urlparse(base_url).netloc
    seen, out = set(), []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")): continue
        anchor = (a.get_text() or "").strip().lower()
        if not any(re.search(p, (href + " " + anchor).lower()) for p in MENU_LINK_PATTERNS): continue
        full = urljoin(base_url, href)
        same_domain = urlparse(full).netloc == base_domain
        is_pdf = full.lower().split("?")[0].endswith(".pdf")
        if not (same_domain or is_pdf): continue
        if full == base_url or full in seen: continue
        seen.add(full); out.append(full)
        if len(out) >= 3: break
    return out

print("§4 — RYAN")

# Test 4.1: extract from simple menu page strips nav/footer, keeps menu items
html = F["sample_html_pages"]["simple_menu_page"]
text = _extract_html_text(html)
has_menu = "Cheese Slice" in text and "Pepperoni Slice" in text
no_nav = "Home | About | Contact" not in text or text.count("Home") <= 1
record("4.1 HTML extraction keeps menu items", has_menu, f"{len(text)} chars extracted")
record("4.2 HTML extraction strips nav/footer", no_nav, "")

# Test 4.3: menu link finder picks up internal /menu and PDF
html = F["sample_html_pages"]["html_with_menu_link"]
links = _find_menu_links(html, "https://example.com/")
has_menu_link = any("/menu" in l for l in links)
has_pdf_link = any(l.endswith(".pdf") for l in links)
no_external = not any("other-site.com" in l for l in links)
record("4.3 finds same-domain /menu link", has_menu_link, f"links: {links}")
record("4.4 finds PDF menu link", has_pdf_link, "")
record("4.5 excludes external-domain links (non-PDF)", no_external, "")

# Test 4.6: cache layer roundtrips
test_url = "https://test.example/menu"
test_data = {"url": test_url, "status": "success", "content": "test content"}
_cache_path(test_url).write_text(json.dumps(test_data))
loaded = json.loads(_cache_path(test_url).read_text())
record("4.6 cache write+read roundtrips", loaded == test_data, "")


## 5. Leytisha — Agent tests

Tests structuring, retrieval, and (if API key set) ethics + reflection.


In [ ]:
print("§5 — LEYTISHA")

def structure_record(enriched: dict) -> dict:
    e = enriched.get("enrichment", {}) or {}
    return {
        "place_id": enriched.get("place_id"), "name": enriched.get("name"),
        "address": enriched.get("address"), "city": enriched.get("city"),
        "rating": enriched.get("rating"), "price_level": enriched.get("price_level"),
        "website_url": enriched.get("website_url"),
        "cuisine_type": e.get("cuisine_type") or "Unknown",
        "menu_items": e.get("menu_items") or [],
        "price_range": e.get("price_range") or "$$",
        "menu_available_online": bool(e.get("menu_available_online", False)),
        "scrape_failed": bool(e.get("scrape_failed", False)),
    }

def simple_retrieve(query: str, records: list, k: int = 5) -> list:
    q = query.lower()
    scored = []
    for r in records:
        haystack = " ".join([r.get("name", ""), r.get("city", ""),
                             r.get("cuisine_type", ""), r.get("price_range", ""),
                             " ".join((it.get("name", "") if isinstance(it, dict) else str(it))
                                      for it in r.get("menu_items", []))]).lower()
        score = sum(1 for token in q.split() if token in haystack)
        if score > 0: scored.append((score, r))
    scored.sort(key=lambda x: -x[0])
    return [r for _, r in scored[:k]] or records[:k]

# Test 5.1: structuring fills in defaults when enrichment is incomplete
partial = {**F["normalized_restaurants"][0], "enrichment": {}}
result = structure_record(partial)
record("5.1 structuring defaults: cuisine_type='Unknown' when missing", result["cuisine_type"] == "Unknown")
record("5.2 structuring defaults: menu_items=[] when missing", result["menu_items"] == [])
record("5.3 structuring defaults: price_range='$$' when missing", result["price_range"] == "$$")

# Test 5.4: retrieve hits expected top result for clear queries
records = F["enriched_restaurants"]
for tq in F["test_queries"]:
    if "expected_top_name" in tq:
        hits = simple_retrieve(tq["query"], records, k=3)
        passed = hits and hits[0]["name"] == tq["expected_top_name"]
        record(f"5.4 retrieve {tq['query']!r} → top={tq['expected_top_name']}",
               passed, f"got {hits[0]['name'] if hits else 'nothing'}")


In [ ]:
# --- Optional: ethics + reflection (requires OPENAI_API_KEY) ---
if not os.environ.get("OPENAI_API_KEY"):
    print("\n  ⊘ Ethics + reflection tests SKIPPED (set OPENAI_API_KEY to enable)")
else:
    from openai import OpenAI
    client = OpenAI()

    ETHICS_RUBRIC_PROMPT = """You are an evaluator scoring restaurant recommendation responses on five ethical dimensions.

Score each 1-5 (5 = best). Respond ONLY with this JSON:
{"geographic_fairness": {"score": int, "note": str},
 "price_diversity": {"score": int, "note": str},
 "cuisine_respect": {"score": int, "note": str},
 "transparency": {"score": int, "note": str},
 "faithfulness": {"score": int, "note": str},
 "overall": int, "issues": [str]}"""

    def evaluate_ethics(query, response, retrieved):
        eval_input = (f"USER QUERY:\n{query}\n\nRETRIEVED RECORDS:\n"
                      f"{json.dumps(retrieved, indent=2)}\n\nRESPONSE:\n{response}")
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": ETHICS_RUBRIC_PROMPT},
                      {"role": "user", "content": eval_input}],
            response_format={"type": "json_object"})
        return json.loads(resp.choices[0].message.content)

    records_by_id = {r["place_id"]: r for r in F["enriched_restaurants"]}
    print("\n  Running ethics rubric on test cases...")
    for case in F["ethics_test_cases"]:
        retrieved = [records_by_id[pid] for pid in case["retrieved_records_ids"] if pid in records_by_id]
        try:
            result = evaluate_ethics(case["query"], case["response"], retrieved)
            overall = result.get("overall", 0)
            if "expected_min_overall" in case:
                ok = overall >= case["expected_min_overall"]
                record(f"5.5 ethics[{case['name']}] overall>={case['expected_min_overall']}",
                       ok, f"got {overall}")
            if "expected_max_overall" in case:
                ok = overall <= case["expected_max_overall"]
                record(f"5.6 ethics[{case['name']}] overall<={case['expected_max_overall']}",
                       ok, f"got {overall}")
        except Exception as e:
            record(f"5.* ethics[{case['name']}] threw exception", False, str(e)[:80])


## 6. Toby — ChromaDB tests

Tests embedding-text construction, ingestion, and reranked retrieval. No OpenAI needed (uses local sentence-transformers embeddings).


In [ ]:
import chromadb
from typing import Optional

print("§6 — TOBY")

CHROMA_TEST_DIR = Path("./chroma_test_store")
chroma_client = chromadb.PersistentClient(path=str(CHROMA_TEST_DIR))
TEST_COLLECTION = "test_restaurants"

def build_embedding_text(record: dict) -> str:
    parts = [record.get("name", ""), f"in {record.get('city', '')}",
             f"cuisine: {record.get('cuisine_type', '')}",
             f"price: {record.get('price_range', '')}"]
    items = record.get("menu_items") or []
    if items:
        names = [(it.get("name") if isinstance(it, dict) else str(it)) for it in items]
        parts.append("menu includes: " + ", ".join(names))
    return " | ".join(p for p in parts if p)

def load_records_to_chroma(records, collection_name=TEST_COLLECTION):
    try: chroma_client.delete_collection(collection_name)
    except Exception: pass
    coll = chroma_client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
    coll.add(
        ids=[r["place_id"] for r in records],
        documents=[build_embedding_text(r) for r in records],
        metadatas=[{
            "name": r.get("name", ""), "city": r.get("city", ""),
            "address": r.get("address", ""), "cuisine_type": r.get("cuisine_type", ""),
            "price_range": r.get("price_range", ""),
            "rating": float(r.get("rating") or 0),
            "website_url": r.get("website_url", ""),
            "menu_items_json": json.dumps(r.get("menu_items", []))
        } for r in records])
    return coll

def _rerank(results, prefer_city=None):
    for r in results:
        score = r["distance"]
        rating = r["metadata"].get("rating", 0)
        score -= 0.05 * (rating - 4.0)
        if prefer_city and r["metadata"].get("city", "").lower() == prefer_city.lower():
            score -= 0.15
        r["rerank_score"] = score
    results.sort(key=lambda r: r["rerank_score"])
    return results

def chroma_search(query, k=5, prefer_city=None, collection_name=TEST_COLLECTION):
    coll = chroma_client.get_collection(collection_name)
    raw = coll.query(query_texts=[query], n_results=k * 2)
    results = []
    for i in range(len(raw["ids"][0])):
        results.append({"id": raw["ids"][0][i], "document": raw["documents"][0][i],
                        "metadata": raw["metadatas"][0][i],
                        "distance": raw["distances"][0][i]})
    reranked = _rerank(results, prefer_city=prefer_city)[:k]
    return [{"place_id": r["id"], "name": r["metadata"]["name"],
             "city": r["metadata"]["city"], "cuisine_type": r["metadata"]["cuisine_type"],
             "price_range": r["metadata"]["price_range"], "rating": r["metadata"]["rating"],
             "website_url": r["metadata"]["website_url"],
             "menu_items": json.loads(r["metadata"].get("menu_items_json", "[]")),
             "_rerank_score": round(r["rerank_score"], 4)} for r in reranked]

# Test 6.1: embedding text includes signal-bearing fields
text = build_embedding_text(F["enriched_restaurants"][0])
record("6.1 embedding text includes name", "Dinosaur Bar-B-Que" in text)
record("6.2 embedding text includes city", "Syracuse" in text)
record("6.3 embedding text includes cuisine", "BBQ" in text)
record("6.4 embedding text includes menu items", "Brisket" in text or "Pulled" in text or "Mac" in text)

# Test 6.5: load + retrieve
print("\n  Loading records into Chroma (downloads embedding model first time)...")
load_records_to_chroma(F["enriched_restaurants"])
print("  Loaded.\n")

# Test 6.6: query semantics
hits = chroma_search("smoked meat brisket", k=3)
top = hits[0]["name"] if hits else None
record("6.6 'smoked meat brisket' → top=Dinosaur", top == "Dinosaur Bar-B-Que",
       f"got {top!r}, scores: {[h['_rerank_score'] for h in hits[:3]]}")

# Test 6.7: city preference rerank
hits_no_pref = chroma_search("Italian food", k=3)
hits_albany = chroma_search("Italian food", k=3, prefer_city="Albany")
top_albany = hits_albany[0]["city"] if hits_albany else None
record("6.7 prefer_city='Albany' biases toward Albany", top_albany == "Albany",
       f"got top city {top_albany!r}")


## 7. Full-pipeline integration test (requires OpenAI key)

Runs end-to-end: query → Chroma retrieval → LLM response → ethics rubric → self-reflection.


In [ ]:
print("§7 — FULL PIPELINE")
if not os.environ.get("OPENAI_API_KEY"):
    print("  ⊘ SKIPPED (set OPENAI_API_KEY to enable)")
else:
    # Use Chroma retrieval (Toby) → query agent (Leytisha) → ethics + reflection
    from openai import OpenAI
    client = OpenAI()

    test_query = "Italian food in upstate NY"
    retrieved = chroma_search(test_query, k=4)
    print(f"  Retrieved {len(retrieved)} restaurants for: {test_query!r}")

    # Generate response
    sys_prompt = ("Recommend restaurants from the records below. Mention name, city, "
                  "cuisine, price, rating, and ONE concrete reason per pick. Be concise.")
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": sys_prompt},
                  {"role": "user", "content": f"{test_query}\n\nRecords:\n{json.dumps(retrieved)}"}])
    response_text = resp.choices[0].message.content
    print(f"\n  Response:\n  {response_text[:300]}...")

    # Run ethics on it
    eth_resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": ETHICS_RUBRIC_PROMPT},
                  {"role": "user", "content": f"QUERY: {test_query}\n\nRECORDS:\n{json.dumps(retrieved)}\n\nRESPONSE:\n{response_text}"}],
        response_format={"type": "json_object"})
    ethics = json.loads(eth_resp.choices[0].message.content)
    print(f"\n  Ethics overall: {ethics.get('overall')}/5")
    print(f"  Issues: {ethics.get('issues', [])}")

    record("7.1 full pipeline runs end-to-end", True, "")
    record("7.2 ethics returned valid overall score", isinstance(ethics.get("overall"), int))


## 8. Summary

In [ ]:
print("\n" + "=" * 60)
print("TEST SUMMARY")
print("=" * 60)
total = len(RESULTS)
passed = sum(1 for _, p, _ in RESULTS if p)
failed = total - passed
print(f"Total:  {total}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
if failed:
    print("\nFailures:")
    for name, p, detail in RESULTS:
        if not p:
            print(f"  ✗ {name}  — {detail}")
print("=" * 60)
